# SQL Intermediate Practice Exercises

**Estimated time:** ~7 hours (part of the ~12 hour sql-intermediate level, alongside
`sql-intermediate-guide.ipynb`).

These follow `sql-intermediate-guide.ipynb` in order, and every heading names the **guide section** it
practises. Numbering starts at 2 because guide section 1 is the setup, which the cell below covers.

## How to Use This Notebook

- **Run the setup cell first.** It builds the shop database in memory from `../assets/sql/` and defines `q()`
  and `run()`.
- Write your SQL between the triple quotes, then run the cell with **Shift + Enter**.
- Every cell ends with `assert` checks. You are right when it prints `OK` with no `AssertionError`.
- **Use the exact column names and aliases the instructions ask for**, and the exact `ORDER BY` — the checks
  compare positions, and without a deterministic sort there is no correct answer to compare against.
- Before you write a query with a join in it, say out loud what one row of the result means. At this level most
  wrong answers are right queries at the wrong grain.
- Build the long ones a CTE at a time. Write the first block, `SELECT * FROM` it, check the grain, then add the
  next. That is how the guide's queries were written.
- Revenue means `quantity * unit_price * (1 - discount)` throughout, and unless an exercise says otherwise,
  cancelled orders are excluded.

## Setup — Run This First

Builds the database in memory and defines `q(sql)` and `run(sql)`. Run it once; run it again after any kernel
restart.

In [1]:
import sqlite3
import pandas as pd

SQL = "assets/sql"          # the bundled schema and CSV files
TABLES = ["categories", "customers", "employees", "products",
          "orders", "order_items", "payments"]

con = sqlite3.connect(":memory:")         # the database lives in RAM -- nothing to clean up

with open(f"{SQL}/schema.sql") as f:
    con.executescript(f.read())           # creates the seven empty tables

con.execute("PRAGMA foreign_keys = ON")   # from here on, SQLite enforces the foreign keys

for table in TABLES:
    pd.read_csv(f"{SQL}/{table}.csv").to_sql(table, con, if_exists="append", index=False)
con.commit()


def q(sql):
    """Run a SELECT and hand the result back as a pandas DataFrame."""
    return pd.read_sql_query(sql, con)


def run(sql):
    """Run statements that change data or structure: CREATE, INSERT, UPDATE, DELETE."""
    con.executescript(sql)
    con.commit()


pd.set_option("display.width", 110)
pd.set_option("display.max_rows", 25)

for table in TABLES:
    print(f"{table:12s} {q(f'SELECT COUNT(*) AS n FROM {table}')['n'][0]:>4} rows")

categories      8 rows
customers      60 rows
employees      15 rows
products       40 rows
orders        300 rows
order_items   673 rows
payments      248 rows


## Exercise 2: The Self-Join  *(guide section 2)*

List every employee alongside their manager. Columns:

- `employee` — the employee's `name`
- `role`
- `manager` — their manager's `name`, or `'none'` for the one person who has no manager
- `manager_role` — their manager's `role`, or `'none'`

Order by `employee_id`.

The founder must appear in the result, which decides for you which kind of join to use.

In [5]:
sql = """
SELECT e.name as employee , e.role as role  , coalesce(m.name,'none') as manager, coalesce(m.role,'none') as manager_role
From employees as e left join employees as m
on e.manager_id = m.employee_id
order by e.employee_id
"""

out = q(sql)

assert list(out.columns) == ["employee", "role", "manager", "manager_role"]
assert len(out) == 15
assert out.round(2).values.tolist() == [
    ["Radhika Menon", "Founder", "none", "none"],
    ["Vikram Nair", "Sales Manager", "Radhika Menon", "Founder"],
    ["Sunita Rao", "Sales Manager", "Radhika Menon", "Founder"],
    ["Imran Sheikh", "Support Manager", "Radhika Menon", "Founder"],
    ["Arjun Pillai", "Sales Rep", "Vikram Nair", "Sales Manager"],
    ["Kavya Krishnan", "Sales Rep", "Vikram Nair", "Sales Manager"],
    ["Devendra Joshi", "Sales Rep", "Vikram Nair", "Sales Manager"],
    ["Priya Balan", "Sales Rep", "Sunita Rao", "Sales Manager"],
    ["Nikhil Verma", "Sales Rep", "Sunita Rao", "Sales Manager"],
    ["Farah Qureshi", "Sales Rep", "Sunita Rao", "Sales Manager"],
    ["Sanjay Gupta", "Support Agent", "Imran Sheikh", "Support Manager"],
    ["Meera Iyer", "Support Agent", "Imran Sheikh", "Support Manager"],
    ["Tarun Das", "Support Agent", "Imran Sheikh", "Support Manager"],
    ["Lakshmi Suresh", "Sales Rep", "Vikram Nair", "Sales Manager"],
    ["Omar Farooq", "Sales Rep", "Sunita Rao", "Sales Manager"],
]
print("OK")

OK


## Exercise 3: Row Fan-Out  *(guide section 3)*

The shop's total payments come to a specific number. Prove you can still get it after a join.

1. `fanned` — a single row, single column `paid`: `ROUND(SUM(p.amount), 2)` from `payments p` **joined to**
   `order_items i` on `order_id`. This is the wrong answer, and the check expects the wrong number.
2. `correct` — the same `paid` column, but right: still join `payments` to the item lines, and still report
   `ROUND(SUM(p.amount), 2)`, but collapse `order_items` to one row per order in a CTE first.

Getting the same number as `SELECT SUM(amount) FROM payments` while a join to `order_items` is present is the
whole point.

In [6]:
fanned_sql = """
SELECT ROUND(SUM(p.amount), 2) AS paid
FROM payments p
JOIN order_items i
 ON i.order_id = p.order_id
"""
fanned = q(fanned_sql)

correct_sql = """
WITH item_count AS (
SELECT order_id, COUNT(*) AS lines
FROM order_items
GROUP BY order_id
)
SELECT ROUND(SUM(p.amount), 2) AS paid
FROM payments p
JOIN item_count c
ON c.order_id = p.order_id
"""
correct = q(correct_sql)

assert list(fanned.columns) == ["paid"]
assert len(fanned) == 1
assert fanned.round(2).values.tolist() == [[54971902.5]]
assert list(correct.columns) == ["paid"]
assert len(correct) == 1
assert correct.round(2).values.tolist() == [[19354240]]
print("OK")

OK


## Exercise 4: Anti-Joins  *(guide section 4)*

Find every customer who has **never had an order reach `'delivered'`** — including the nine who never ordered
at all.

Columns `customer_id` and `name`, ordered by `customer_id`.

Use `NOT EXISTS`. A `LEFT JOIN` on status would also work, but only if you put the status condition in the
right place — which is exercise 22.

In [7]:
sql = """
select c.customer_id as customer_id , c.name as name 
from customers c
where not exists ( select 1 
from orders o
where c.customer_id = o.customer_id and o.status='delivered' 
)
order by c.customer_id
"""

out = q(sql)

assert list(out.columns) == ["customer_id", "name"]
assert len(out) == 14
assert out.round(2).values.tolist() == [
    [4, "Pooja Singh"],
    [5, "Varun Chopra"],
    [11, "Deepa Mehta"],
    [16, "Heena Singh"],
    [20, "Ganesh Reddy"],
    [21, "Zoya Bose"],
    [23, "Ojas Mehta"],
    [26, "Bhavya Singh"],
    [30, "Trisha Das"],
    [35, "Omkar Pillai"],
    [41, "Rahul Patel"],
    [46, "Neha Iyer"],
    [55, "Kabir Nair"],
    [60, "Tarun Khan"],
]
print("OK")

OK


## Exercise 5: The NOT IN Trap  *(guide section 5)*

One row, three columns, showing the trap next to its two fixes. Count **products that have never been
ordered**, three different ways:

- `by_not_in` — `NOT IN (SELECT product_id FROM order_items WHERE quantity > 900)`. That inner query returns no
  rows at all, so think about what `NOT IN` over an empty list does before you predict the answer.
- `by_not_in_nullable` — `NOT IN (SELECT o.employee_id FROM orders o)` applied to `p.product_id`. Nonsense as a
  question, but the subquery contains `NULL`s, and that is what you are measuring.
- `by_not_exists` — the honest count, using `NOT EXISTS` against `order_items`.

Use scalar subqueries in a single `SELECT`. The three numbers should not agree, and knowing why each is what it
is means you will never write this bug.

In [ ]:
sql = """
select 
"""

out = q(sql)

assert list(out.columns) == ["by_not_in", "by_not_in_nullable", "by_not_exists"]
assert len(out) == 1
assert out.round(2).values.tolist() == [[40, 0, 2]]
print("OK")

## Exercise 6: Subqueries in WHERE  *(guide section 6)*

Every product that is **cheaper than the average product price** and sits in a category whose name ends in
`'s'` and is one of Audio, Storage or Wearables.

Columns `name`, `price`, `category_id`. Order by `price` descending, then `name`.

Use a **scalar** subquery for the average and a **list** subquery (`IN (SELECT ...)`) for the categories —
match the category by `name`, not by hard-coded id.

In [19]:
sql = """
SELECT
    p.name,
    p.price,
    p.category_id
FROM products p
WHERE p.price < (SELECT AVG(price) FROM products)
  AND p.category_id IN (
      SELECT category_id
      FROM categories
      WHERE name LIKE '%s'
         OR name IN ('Audio', 'Storage', 'Wearables')
  )
ORDER BY p.price DESC, p.name;
"""

out = q(sql)

assert list(out.columns) == ["name", "price", "category_id"]
assert len(out) == 13
assert out.round(2).values.tolist() == [
    ["Halo Studio Headset", 18900, 3],
    ["Halo Over-Ear", 11500, 3],
    ["Vault 2TB SSD", 11200, 6],
    ["Tick Smartwatch", 9900, 8],
    ["Quiet Desk Mic", 7600, 3],
    ["Vault 4TB HDD", 7300, 6],
    ["Echo Buds Pro", 6800, 3],
    ["Vault 1TB SSD", 6400, 6],
    ["Rumble Bluetooth Speaker", 4500, 3],
    ["Echo Buds", 3200, 3],
    ["Tick Fitness Band", 2800, 8],
    ["Carry 256GB Flash Drive", 1950, 6],
    ["Carry 128GB Flash Drive", 1150, 6],
]
print("OK")

AssertionError: 

## Exercise 7: Derived Tables  *(guide section 7)*

Average order value by **customer city**, for non-cancelled orders.

Columns:

- `city` — use `'unknown'` where the customer has none
- `orders` — how many orders
- `avg_order_value` — the average **order** total, rounded to 2 decimals
- `total_revenue` — rounded to 2 decimals

Order by `avg_order_value` descending, then `city`.

This is an average of an average, so it needs two levels: a derived table (or CTE) at one row per order, then a
grouping by city on top. Computing `AVG(quantity * unit_price * ...)` directly gives the average **line** value,
which is a different number.

In [23]:
sql = """
with cte as (
select o.order_id , o.customer_id , sum(i.quantity * i.unit_price * (1 - i.discount)) AS order_total 
from orders o join order_items i 
on o.order_id = i.order_id
where o.status <> 'cancelled'
group by o.order_id , o.customer_id)
SELECT
    COALESCE(c.city, 'unknown') AS city,
    COUNT(*) AS orders,
    ROUND(AVG(ot.order_total), 2) AS avg_order_value,
    ROUND(SUM(ot.order_total), 2) AS total_revenue
FROM cte AS ot
LEFT JOIN customers AS c
    ON ot.customer_id = c.customer_id
GROUP BY COALESCE(c.city, 'unknown')
ORDER BY avg_order_value DESC, city

"""

out = q(sql)

assert list(out.columns) == ["city", "orders", "avg_order_value", "total_revenue"]
assert len(out) == 11
assert out.round(2).values.tolist() == [
    ["Mumbai", 67, 103315.6, 6922145],
    ["Hyderabad", 24, 101445, 2434680],
    ["Kochi", 20, 88644, 1772880],
    ["unknown", 33, 81648.41, 2694397.5],
    ["Ahmedabad", 10, 71225, 712250],
    ["Jaipur", 12, 67035, 804420],
    ["Chennai", 57, 64683.86, 3686980],
    ["Delhi", 9, 63846.94, 574622.5],
    ["Pune", 14, 59596.79, 834355],
    ["Kolkata", 16, 53255.47, 852087.5],
    ["Bengaluru", 20, 40405.75, 808115],
]
print("OK")

OK


## Exercise 8: Correlated Subqueries  *(guide section 8)*

Every product priced **above the average price of its own category**.

Columns:

- `name`
- `category_id`
- `price`
- `category_avg` — the average price of that product's category, rounded to 2 decimals
- `times_ordered` — how many `order_items` rows mention this product

Order by `category_id`, then `price` descending.

Both derived columns and the filter are correlated subqueries. You will rewrite this with a window function in
exercise 12 — keep your answer to compare.

In [24]:
sql = """
select
p.name,
p.category_id,
p.price,
round((
select avg(p2.price)
from products p2
where p2.category_id = p.category_id
), 2) as category_avg,
(
select count(*)
from order_items oi
where oi.product_id = p.product_id
) as times_ordered
from products p
where p.price > (
select avg(p3.price)
from products p3
where p3.category_id = p.category_id
)
order by p.category_id, p.price desc
"""

out = q(sql)

assert list(out.columns) == ["name", "category_id", "price", "category_avg", "times_ordered"]
assert len(out) == 17
assert out.round(2).values.tolist()[0] == ["Vega Book 16 Studio", 1, 142000, 94000, 15]
assert out.round(2).values.tolist()[-1] == ["Tick Watch Ultra", 8, 26500, 13066.67, 11]
assert round(float(out["category_id"].sum()), 2) == 71
print("OK")

OK


## Exercise 9: EXISTS and NOT EXISTS Together  *(guide section 9)*

Customers who have bought from **Audio** (category 3) but have **never** bought a **Laptop** (category 1).

Columns `customer_id` and `name`, ordered by `customer_id`.

One `EXISTS` and one `NOT EXISTS` in the same `WHERE`. Each customer must appear exactly once, however many
audio products they bought — which is the reason this is not a join.

In [54]:
sql = """
SELECT
    c.customer_id,
    c.name
FROM customers c
WHERE EXISTS (
    SELECT 1
    FROM orders o
    JOIN order_items oi
        ON oi.order_id = o.order_id
    JOIN products p
        ON p.product_id = oi.product_id
    WHERE o.customer_id = c.customer_id
      AND p.category_id = 3
)
AND NOT EXISTS (
    SELECT 1
    FROM orders o
    JOIN order_items oi
        ON oi.order_id = o.order_id
    JOIN products p
        ON p.product_id = oi.product_id
    WHERE o.customer_id = c.customer_id
      AND p.category_id = 1
)
ORDER BY c.customer_id;
"""

out = q(sql)

assert list(out.columns) == ["customer_id", "name"]
assert len(out) == 11
assert out.round(2).values.tolist() == [
    [3, "Mohit Pillai"],
    [15, "Kabir Gupta"],
    [20, "Ganesh Reddy"],
    [28, "Tarun Khan"],
    [32, "Ishita Nair"],
    [33, "Eshan Gupta"],
    [40, "Jyoti Nair"],
    [44, "Ekta Menon"],
    [46, "Neha Iyer"],
    [49, "Zara Reddy"],
    [53, "Ibrahim Bose"],
]
print("OK")

OK


## Exercise 10: Set Operators  *(guide section 10)*

A breakdown of orders by `status` with a grand total stacked underneath.

Columns `label` and `orders`. The first part is one row per status (`label` is the status); the second part is a
single row with `label` set to the literal `'ALL STATUSES'`.

Order the combined result by `orders` descending, then `label`.

Use `UNION ALL`, not `UNION` — and remember the `ORDER BY` belongs to the whole statement, at the very end.

In [32]:
sql = """
select status as label , count(*) as orders 
from orders 
group by status
union all
select 'ALL STATUSES' , count(*) as orders
from orders
order by orders desc , label

"""

out = q(sql)

assert list(out.columns) == ["label", "orders"]
assert len(out) == 6
assert out.round(2).values.tolist() == [
    ["ALL STATUSES", 300],
    ["delivered", 176],
    ["shipped", 50],
    ["placed", 39],
    ["cancelled", 18],
    ["returned", 17],
]
print("OK")

OK


## Exercise 11: CTE Chains  *(guide section 11)*

The **top 10 customers by lifetime value**, written as a chain of CTEs.

Columns:

- `name`
- `city` — `'unknown'` where missing
- `orders` — number of non-cancelled orders
- `lifetime_value` — total revenue, rounded to 2 decimals
- `avg_order` — `lifetime_value / orders`, rounded to 2 decimals
- `last_order` — most recent `order_date`

Order by `lifetime_value` descending, then `name`. Limit 10.

Write it as at least two CTEs: one at the grain of one row per order, one at one row per customer. Doing it in a
single query means counting orders after a fan-out.

In [53]:
sql = """
WITH order_revenue AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_date,
        SUM(i.quantity * i.unit_price * (1 - i.discount)) AS order_total
    FROM orders o
    JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status != 'cancelled'
    GROUP BY o.order_id
),
customer_totals AS (
    SELECT
        customer_id,
        COUNT(*) AS orders,
        SUM(order_total) AS lifetime_value,
        MAX(order_date) AS last_order
    FROM order_revenue
    GROUP BY customer_id
)
SELECT
    c.name,
    COALESCE(c.city, 'unknown') AS city,
    ct.orders,
    ROUND(ct.lifetime_value, 2) AS lifetime_value,
    ROUND(ct.lifetime_value / ct.orders, 2) AS avg_order,
    ct.last_order
FROM customer_totals ct
JOIN customers c ON c.customer_id = ct.customer_id
ORDER BY lifetime_value DESC, name
LIMIT 10
"""

out = q(sql)

assert list(out.columns) == ["name", "city", "orders", "lifetime_value", "avg_order", "last_order"]
assert len(out) == 10
assert out.round(2).values.tolist() == [
    ["Hema Khan", "Mumbai", 19, 2835360, 149229.47, "2024-11-20"],
    ["Neha Reddy", "Mumbai", 19, 1539182.5, 81009.61, "2024-12-28"],
    ["Zara Mehta", "Hyderabad", 12, 1257470, 104789.17, "2024-12-09"],
    ["Varun Pillai", "unknown", 9, 1002250, 111361.11, "2024-12-03"],
    ["Parvati Chopra", "Kochi", 6, 826850, 137808.33, "2024-09-02"],
    ["Nisha Nair", "Hyderabad", 9, 805260, 89473.33, "2024-12-04"],
    ["Yash Bose", "Kochi", 10, 791490, 79149, "2024-12-25"],
    ["Janaki Patel", "Jaipur", 10, 783195, 78319.5, "2024-12-31"],
    ["Manoj Menon", "Mumbai", 8, 781007.5, 97625.94, "2024-11-19"],
    ["Harish Reddy", "unknown", 11, 759455, 69041.36, "2024-12-11"],
]
print("OK")

OK


## Exercise 12: Window Functions  *(guide section 12)*

The same question as exercise 8 — products priced above their category average — but with **no subqueries at
all**.

Columns `name`, `category_id`, `price`, `category_avg` (rounded to 2), `products_in_category`.
Order by `category_id`, then `price` descending.

Compute the category average with `AVG(price) OVER (PARTITION BY category_id)` in a CTE, then filter in the
outer query. You cannot filter on a window function in the same `SELECT` that computes it.

The rows should match your answer to exercise 8 exactly. One pass instead of three.

In [34]:
sql = """
with cte as (
select name , category_id , price , avg(price) over (partition by category_id) as category_avg , count(*) over (partition by category_id) as products_in_category
from products
)
select name , category_id , price , category_avg , products_in_category
from cte 
where price>category_avg
order by category_id  , price desc
"""

out = q(sql)

assert list(out.columns) == ["name", "category_id", "price", "category_avg", "products_in_category"]
assert len(out) == 17
assert out.round(2).values.tolist()[0] == ["Vega Book 16 Studio", 1, 142000, 94000, 5]
assert out.round(2).values.tolist()[-1] == ["Tick Watch Ultra", 8, 26500, 13066.67, 3]
assert round(float(out["category_id"].sum()), 2) == 71
print("OK")

OK


## Exercise 13: Top N per Group  *(guide section 13)*

The **three best-selling products by revenue in each category**, excluding cancelled orders.

Columns:

- `category` — the category's `name`
- `product` — the product's `name`
- `units` — total quantity sold
- `revenue` — rounded to 2 decimals
- `rn` — the product's rank within its category, 1 to 3

Order by `category`, then `rn`.

Use `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY revenue DESC, product)` — the product name as a tiebreaker so
the answer is reproducible — and filter `rn <= 3` outside the CTE that computes it.

In [38]:
sql = """
with sales as (
    select
    p.category_id,
     p.name as product,
    sum(oi.quantity) as units,
    sum(oi.quantity * oi.unit_price * (1 - oi.discount)) as revenue
    from products p
    join order_items oi
     on p.product_id = oi.product_id
    join orders o
    on o.order_id = oi.order_id
    where o.status <> 'cancelled'
    group by p.category_id, p.name
),
ranked as (
    select
     category_id, product,
     units,        round(revenue, 2) as revenue,        row_number() over (
     partition by category_id
     order by revenue desc, product
      ) as rn
    from sales
)
select
    c.name as category,
    r.product,    r.units, r.revenue,r.rn
from ranked r
join categories c
on c.category_id = r.category_id
where r.rn <= 3
order by category, rn;

"""

out = q(sql)

assert list(out.columns) == ["category", "product", "units", "revenue", "rn"]
assert len(out) == 24
assert out.round(2).values.tolist() == [
    ["Accessories", "Clack Mechanical Keyboard", 37, 176890, 1],
    ["Accessories", "Clack Mini Keyboard", 25, 87480, 2],
    ["Accessories", "Anchor 100W Charger", 24, 68730, 3],
    ["Audio", "Halo Studio Headset", 21, 385560, 1],
    ["Audio", "Echo Buds Pro", 39, 259080, 2],
    ["Audio", "Halo Over-Ear", 21, 234600, 3],
    ["Cameras", "Frame Mirrorless Body", 16, 1328700, 1],
    ["Cameras", "Frame Compact Camera", 26, 867000, 2],
    ["Cameras", "Frame 50mm Lens", 26, 386540, 3],
    ["Laptops", "Vega Book 16 Studio", 23, 3102700, 1],
    ["Laptops", "Nimbus Air Laptop", 18, 2082700, 2],
    ["Laptops", "Vega Book 13", 28, 1463400, 3],
    ["Monitors", "Wide 34 Curved Monitor", 24, 1324400, 1],
    ["Monitors", "Clarity 32 4K Monitor", 31, 1126700, 2],
    ["Monitors", "Clarity 27 QHD Monitor", 23, 506800, 3],
    ["Phones", "Orbit Pro Phone", 26, 1219200, 1],
    ["Phones", "Orbit Ultra Phone", 16, 1196850, 2],
    ["Phones", "Orbit One Phone", 24, 623700, 3],
    ["Storage", "Vault 2TB SSD", 37, 408800, 1],
    ["Storage", "Vault 4TB HDD", 37, 259880, 2],
    ["Storage", "Vault 1TB SSD", 37, 227840, 3],
    ["Wearables", "Tick Watch Ultra", 18, 449175, 1],
    ["Wearables", "Tick Smartwatch", 18, 178665, 2],
    ["Wearables", "Tick Fitness Band", 25, 68460, 3],
]
print("OK")

OK


## Exercise 14: LAG and Month-over-Month  *(guide section 14)*

Monthly revenue for **2024** with the change on the previous month.

Columns:

- `month` — `strftime('%Y-%m', o.order_date)`
- `revenue` — rounded to 2
- `prev_month` — the previous month's revenue, rounded to 2
- `change` — `revenue - prev_month`, rounded to 2
- `pct_change` — the change as a percentage of `prev_month`, rounded to 1

Order by `month`.

Restrict to 2024 **inside** the CTE that builds the monthly series, so January 2024 has no previous month and
its `prev_month`, `change` and `pct_change` are all missing. The checks expect that.

In [39]:
sql = """
with monthly as (
select strftime('%Y-%m', o.order_date) as month,sum(oi.quantity * oi.unit_price * (1 - oi.discount)) as revenue
from orders o
join order_items oi
on oi.order_id = o.order_id
where o.status <> 'cancelled'
and o.order_date >= '2024-01-01'
and o.order_date < '2025-01-01'
group by strftime('%Y-%m', o.order_date)
),
lagged as (
select month,revenue,lag(revenue) over (order by month) as prev_month
from monthly
)
select month,
round(revenue, 2) as revenue,
round(prev_month, 2) as prev_month,
round(revenue - prev_month, 2) as change,
round((revenue - prev_month) * 100.0 / prev_month, 1) as pct_change
from lagged
order by month;
"""

out = q(sql)

assert list(out.columns) == ["month", "revenue", "prev_month", "change", "pct_change"]
assert len(out) == 12
assert out.round(2).fillna("<NULL>").values.tolist() == [
    ["2024-01", 414130, "<NULL>", "<NULL>", "<NULL>"],
    ["2024-02", 315950, 414130, -98180, -23.7],
    ["2024-03", 983710, 315950, 667760, 211.3],
    ["2024-04", 321250, 983710, -662460, -67.3],
    ["2024-05", 1246130, 321250, 924880, 287.9],
    ["2024-06", 920957.5, 1246130, -325172.5, -26.1],
    ["2024-07", 796725, 920957.5, -124232.5, -13.5],
    ["2024-08", 1690255, 796725, 893530, 112.2],
    ["2024-09", 2164965, 1690255, 474710, 28.1],
    ["2024-10", 1250365, 2164965, -914600, -42.2],
    ["2024-11", 2328947.5, 1250365, 1078582.5, 86.3],
    ["2024-12", 1945537.5, 2328947.5, -383410, -16.5],
]
print("OK")

OK


## Exercise 15: Running Totals and Moving Averages  *(guide section 15)*

Monthly revenue across the whole two years, with three window columns:

- `month`, `revenue` (rounded to 2)
- `running_total` — cumulative revenue up to and including this month, rounded to 2
- `moving_avg_3m` — the average of this month and the two before it, rounded to 2
- `pct_of_total` — the running total as a percentage of the **grand** total, rounded to 1

Order by `month`.

The last two need explicit frames. `moving_avg_3m` is `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`; the grand
total in the denominator needs `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`, because with an
`ORDER BY` present the default frame would give you another running total.

In [55]:
sql = """
WITH monthly AS (
    SELECT
        strftime('%Y-%m', o.order_date) AS month,
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount)
        ) AS revenue
    FROM orders o
    JOIN order_items oi
        ON oi.order_id = o.order_id
    WHERE o.status != 'cancelled'
    GROUP BY strftime('%Y-%m', o.order_date)
)

SELECT
    month,

    ROUND(revenue, 2) AS revenue,

    ROUND(
        SUM(revenue) OVER (
            ORDER BY month
        ),
        2
    ) AS running_total,

    ROUND(
        AVG(revenue) OVER (
            ORDER BY month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ),
        2
    ) AS moving_avg_3m,

    ROUND(
        100.0 * 
        SUM(revenue) OVER (
            ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
        /
        SUM(revenue) OVER (
            ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING
                 AND UNBOUNDED FOLLOWING
        ),
        1
    ) AS pct_of_total

FROM monthly
ORDER BY month;
"""

out = q(sql)

assert list(out.columns) == ["month", "revenue", "running_total", "moving_avg_3m", "pct_of_total"]
assert len(out) == 24
assert out.round(2).values.tolist() == [
    ["2023-01", 178950, 178950, 178950, 0.8],
    ["2023-02", 447725, 626675, 313337.5, 2.8],
    ["2023-03", 704500, 1331175, 443725, 6],
    ["2023-04", 739490, 2070665, 630571.67, 9.4],
    ["2023-05", 442100, 2512765, 628696.67, 11.4],
    ["2023-06", 912610, 3425375, 698066.67, 15.5],
    ["2023-07", 393290, 3818665, 582666.67, 17.3],
    ["2023-08", 1205505, 5024170, 837135, 22.7],
    ["2023-09", 527510, 5551680, 708768.33, 25.1],
    ["2023-10", 439365, 5991045, 724126.67, 27.1],
    ["2023-11", 973450, 6964495, 646775, 31.5],
    ["2023-12", 753515, 7718010, 722110, 34.9],
    ["2024-01", 414130, 8132140, 713698.33, 36.8],
    ["2024-02", 315950, 8448090, 494531.67, 38.2],
    ["2024-03", 983710, 9431800, 571263.33, 42.7],
    ["2024-04", 321250, 9753050, 540303.33, 44.1],
    ["2024-05", 1246130, 10999180, 850363.33, 49.8],
    ["2024-06", 920957.5, 11920137.5, 829445.83, 53.9],
    ["2024-07", 796725, 12716862.5, 987937.5, 57.6],
    ["2024-08", 1690255, 14407117.5, 1135979.17, 65.2],
    ["2024-09", 2164965, 16572082.5, 1550648.33, 75],
    ["2024-10", 1250365, 17822447.5, 1701861.67, 80.7],
    ["2024-11", 2328947.5, 20151395, 1914759.17, 91.2],
    ["2024-12", 1945537.5, 22096932.5, 1841616.67, 100],
]
print("OK")

OK


## Exercise 16: Quartiles with NTILE  *(guide section 16)*

Split the customers who have ordered into **four buckets by total spend**, biggest spenders in quartile 1, then
summarise each bucket.

Columns:

- `quartile`
- `customers` — how many are in it
- `min_spend`, `max_spend`, `total_spend` — all rounded to 2
- `pct_of_revenue` — this quartile's share of all revenue, rounded to 1

Order by `quartile`.

Cancelled orders are excluded. The nine customers who never ordered are not in the population at all — they
have no spend to rank.

In [58]:
sql = """
WITH customer_spend AS (
    SELECT
        o.customer_id,
        SUM(
            oi.quantity * oi.unit_price * (1 - oi.discount)
        ) AS spend
    FROM orders o
    JOIN order_items oi
        ON oi.order_id = o.order_id
    WHERE o.status != 'cancelled'
    GROUP BY o.customer_id
),

bucketed AS (
    SELECT
        customer_id,
        spend,
        NTILE(4) OVER (ORDER BY spend DESC) AS quartile
    FROM customer_spend
)

SELECT
    quartile,
    COUNT(*) AS customers,
    ROUND(MIN(spend), 2) AS min_spend,
    ROUND(MAX(spend), 2) AS max_spend,
    ROUND(SUM(spend), 2) AS total_spend,
    ROUND(
        100.0 * SUM(spend) /
        SUM(SUM(spend)) OVER (),
        1
    ) AS pct_of_revenue
FROM bucketed
GROUP BY quartile
ORDER BY quartile;
"""

out = q(sql)

assert list(out.columns) == ["quartile", "customers", "min_spend", "max_spend", "total_spend", "pct_of_revenue"]
assert len(out) == 4
assert out.round(2).values.tolist() == [
    [1, 13, 551370, 2835360, 13320430, 60.3],
    [2, 13, 335300, 540040, 5661685, 25.6],
    [3, 13, 103480, 320425, 2567110, 11.6],
    [4, 12, 1105, 95395, 547707.5, 2.5],
]
print("OK")

OK


## Exercise 17: Conditional Aggregation  *(guide section 17)*

A cross-tab: one row per **month of 2024**, one column per **channel**, holding revenue. Cancelled orders
excluded.

Columns `month`, `web`, `app`, `store`, `phone`, `total` — every revenue column rounded to 2 decimals.
Order by `month`.

Fix the grain first: a CTE at one row per order carrying its month, channel and total. Then
`SUM(CASE WHEN channel = 'web' THEN order_total ELSE 0 END)` for each column. The four channel columns must add
up to `total`, which is your check that no order was counted twice.

In [41]:
sql = """
with cte as (
select o.channel as channel , strftime('%Y-%m',o.order_date) as month , sum(i.quantity*i.unit_price*(1-i.discount)) as revenue
from order_items as i join orders as o 
on i.order_id = o.order_id
where strftime('%Y',o.order_date) = '2024' and o.status <> 'cancelled'
group by strftime('%Y-%m',o.order_date) , o.channel 
)
select
month,
round(sum(case when channel = 'web' then revenue else 0 end), 2) as web,
round(sum(case when channel = 'app' then revenue else 0 end), 2) as app,
round(sum(case when channel = 'store' then revenue else 0 end), 2) as store,
round(sum(case when channel = 'phone' then revenue else 0 end), 2) as phone,
round(sum(revenue), 2) as total
from cte
group by month
order by month
"""

out = q(sql)

assert list(out.columns) == ["month", "web", "app", "store", "phone", "total"]
assert len(out) == 12
assert out.round(2).values.tolist() == [
    ["2024-01", 133270, 3240, 264300, 13320, 414130],
    ["2024-02", 63000, 127230, 125720, 0, 315950],
    ["2024-03", 281615, 602350, 98640, 1105, 983710],
    ["2024-04", 83650, 155000, 82600, 0, 321250],
    ["2024-05", 154560, 60900, 666370, 364300, 1246130],
    ["2024-06", 290290, 70480, 556980, 3207.5, 920957.5],
    ["2024-07", 182185, 437775, 50350, 126415, 796725],
    ["2024-08", 964640, 202445, 430170, 93000, 1690255],
    ["2024-09", 601915, 447070, 613980, 502000, 2164965],
    ["2024-10", 535827.5, 201442.5, 473985, 39110, 1250365],
    ["2024-11", 1063622.5, 273150, 282760, 709415, 2328947.5],
    ["2024-12", 430200, 891982.5, 270455, 352900, 1945537.5],
]
print("OK")

OK


## Exercise 18: Months Between Two Dates  *(guide section 18)*

For every customer who has ordered, how long their relationship with the shop has been running.

Columns:

- `customer_id`
- `first_month` — `strftime('%Y-%m', ...)` of their first non-cancelled order
- `last_month` — the same for their most recent one
- `orders` — how many non-cancelled orders
- `span_months` — whole months between the two, using the `year * 12 + month` arithmetic from the guide

Order by `span_months` descending, then `customer_id`. Limit 12.

`strftime` returns text, so wrap each part in `CAST(... AS INTEGER)` before doing arithmetic on it.

In [42]:
sql = """
with customer_orders as (
    select        customer_id,
    min(order_date) as first_date,
     max(order_date) as last_date,
        count(*) as orders
    from orders
    where status <> 'cancelled'
    group by customer_id
)
select
    customer_id,
    strftime('%Y-%m', first_date) as first_month,
    strftime('%Y-%m', last_date) as last_month,
    orders,(cast(strftime('%Y', last_date) as integer) * 12+ cast(strftime('%m', last_date) as integer)
    )-(cast(strftime('%Y', first_date) as integer) * 12
        + cast(strftime('%m', first_date) as integer)
    ) as span_months
from customer_orders
order by span_months desc, customer_id
limit 12;
"""

out = q(sql)

assert list(out.columns) == ["customer_id", "first_month", "last_month", "orders", "span_months"]
assert len(out) == 12
assert out.round(2).values.tolist() == [
    [25, "2023-01", "2024-12", 7, 23],
    [34, "2023-01", "2024-12", 12, 23],
    [54, "2023-01", "2024-12", 2, 23],
    [6, "2023-01", "2024-11", 19, 22],
    [9, "2023-02", "2024-12", 12, 22],
    [13, "2023-02", "2024-12", 7, 22],
    [18, "2023-02", "2024-12", 10, 22],
    [27, "2023-02", "2024-12", 12, 22],
    [19, "2023-03", "2024-12", 9, 21],
    [33, "2023-02", "2024-11", 7, 21],
    [43, "2023-03", "2024-12", 9, 21],
    [59, "2023-03", "2024-12", 19, 21],
]
print("OK")

OK


## Exercise 19: Views  *(guide section 19)*

Create a view called `paid_orders` holding one row per **order that has a payment**, with these columns:

`order_id`, `customer_id`, `order_date`, `channel`, `amount` (from `payments`), `method`.

Then query it: `by_method` should be one row per payment `method` with `payments` (the count) and `total`
(`ROUND(SUM(amount), 2)`), ordered by `total` descending then `method`.

Start with `DROP VIEW IF EXISTS paid_orders;` so the cell can be run twice. The view is a join between `orders`
and `payments` — no aggregation, so no fan-out to worry about.

In [48]:
run("""
drop view if exists paid_orders;

create view paid_orders as
select
    o.order_id,
    o.customer_id,
    o.order_date,
    o.channel,
    p.amount,
    p.method
from orders o
join payments p
    on o.order_id = p.order_id;
""")

by_method = q("""select
    method,
    count(*) as payments,
    round(sum(amount), 2) as total
from paid_orders
group by method
order by total desc, method
""")

assert q("SELECT COUNT(*) AS n FROM paid_orders")["n"][0] == 248, "one row per payment"
assert list(q("SELECT * FROM paid_orders LIMIT 1").columns) == ["order_id", "customer_id", "order_date", "channel", "amount", "method"]
assert list(by_method.columns) == ["method", "payments", "total"]
assert len(by_method) == 5
assert by_method["payments"].sum() == 248
assert round(float(by_method["total"].sum()), 2) == round(float(q("SELECT SUM(amount) AS s FROM payments")["s"][0]), 2)
assert by_method["total"].is_monotonic_decreasing, "order by total descending"
print("OK")

OK


## Exercise 20: A Data Quality Report Card  *(guide section 20)*

One stacked query producing a report card. Columns `check_name` and `n`, in exactly this order of rows:

1. `'customers'` — total customers
2. `'customers with no city'`
3. `'duplicate emails'` — how many email addresses are used by more than one customer
4. `'orders with no payment'`
5. `'products never ordered'`
6. `'order items with no parent order'`

Do **not** add an `ORDER BY` — `UNION ALL` preserves the order the parts are written in, and the checks expect
that order.

Row 3 needs a `GROUP BY ... HAVING COUNT(*) > 1` wrapped in a `COUNT(*)`. Row 6 should come out at zero; a
non-zero answer there would mean the foreign keys had failed.

In [49]:
sql = """
select 'customers' as check_name , count(*) as n 
from customers
union all 
select 'customers with no city' , count(*) 
from customers
where city is null 
union all 
select 'duplicate emails' , count(*)
from (select * from 
customers
group by email
having count(*) > 1)
union all 
select 'orders with no payment' ,count(*)
from (select *
from orders as o left join payments as p 
on o.order_id = p.order_id
where p.order_id is null)
union all 
select 'products never ordered' , count(*)
from (select *
from products p left join order_items o 
on p.product_id = o.product_id
where o.product_id is null)
union all
select 'order items with no parent order', count(*)
from (
    select i.order_id
    from order_items i
    left join orders o
        on i.order_id = o.order_id
    where o.order_id is null
)
"""

out = q(sql)

assert list(out.columns) == ["check_name", "n"]
assert len(out) == 6
assert out.round(2).values.tolist() == [
    ["customers", 60],
    ["customers with no city", 5],
    ["duplicate emails", 2],
    ["orders with no payment", 52],
    ["products never ordered", 2],
    ["order items with no parent order", 0],
]
print("OK")

OK


## Exercise 21: Bulk Load and Upsert  *(guide section 21)*

Build a summary table and make the load idempotent.

1. Create `product_summary` with `product_id INTEGER PRIMARY KEY`, `units INTEGER NOT NULL`,
   `revenue REAL NOT NULL`. Drop it first so the cell can be re-run.
2. Fill it with `INSERT ... SELECT` — one row for **every** product, including the two that have never been
   ordered, which must get `0` and `0.0` rather than `NULL`. Exclude cancelled orders.
   `revenue` is `ROUND(SUM(...), 2)`.
3. Run the **same** `INSERT ... SELECT` a second time with an `ON CONFLICT (product_id) DO UPDATE` clause, so it
   updates instead of failing. The table must still have 40 rows afterwards.

A `LEFT JOIN` from `products` and `COALESCE` are what keep the never-ordered products in with zeros.

In [51]:
run("""
drop table if exists product_summary;

create table product_summary (
    product_id integer primary key,
    units integer not null,
    revenue real not null
);

insert into product_summary (product_id, units, revenue)
select
    p.product_id,
    coalesce(sum(oi.quantity), 0),
    round(coalesce(sum(oi.quantity * oi.unit_price * (1 - oi.discount)), 0), 2)
from products p
left join order_items oi
    on p.product_id = oi.product_id
left join orders o
    on oi.order_id = o.order_id
   and o.status <> 'cancelled'
group by p.product_id;
""")

run("""
insert into product_summary (product_id, units, revenue)
select
    p.product_id,
    coalesce(sum(x.quantity), 0),
    round(coalesce(sum(x.quantity * x.unit_price * (1 - x.discount)), 0), 2)
from products p
left join (
    select
        i.product_id,
        i.quantity,
        i.unit_price,
        i.discount
    from order_items i
    join orders o
        on i.order_id = o.order_id
    where o.status <> 'cancelled'
) x
    on p.product_id = x.product_id
group by p.product_id;
""")

summary = q("SELECT * FROM product_summary ORDER BY revenue DESC, product_id")

assert list(summary.columns) == ["product_id", "units", "revenue"]
assert len(summary) == 40, "every product, including the two never ordered"
assert (summary["units"] == 0).sum() == 2, "the two never-ordered products should be present with 0"
assert summary["revenue"].notna().all(), "no NULLs -- COALESCE the never-ordered rows to 0"
assert round(float(summary["revenue"].sum()), 2) == round(float(q("""SELECT SUM(i.quantity * i.unit_price * (1 - i.discount)) AS r FROM order_items i JOIN orders o ON o.order_id = i.order_id WHERE o.status <> 'cancelled'""")["r"][0]), 2)
assert int(summary["units"].sum()) == int(q("""SELECT SUM(i.quantity) AS u FROM order_items i JOIN orders o ON o.order_id = i.order_id WHERE o.status <> 'cancelled'""")["u"][0])
print("OK")

IntegrityError: UNIQUE constraint failed: product_summary.product_id

## Exercise 22: Where the Condition Goes  *(guide section 23)*

One row showing the difference the `ON`/`WHERE` choice makes. Join `customers` to `orders` with a `LEFT JOIN`
and produce four counts:

- `cond_in_where` — rows when `o.status = 'delivered'` sits in the `WHERE`
- `cond_in_on` — rows when the same condition sits in the `ON`
- `count_star` — `COUNT(*)` over a plain `LEFT JOIN` with no status condition
- `count_orders` — `COUNT(o.order_id)` over that same plain join

Use four scalar subqueries in one `SELECT`. The gaps between these numbers are the two most common quiet bugs
at this level.

In [59]:
sql = """
select 
(select count(*)
from customers c left join orders o 
on c.customer_id = o.customer_id
where o.status = 'delivered') as cond_in_where,
(select count(*)
from customers c left join orders o 
on c.customer_id = o.customer_id
and o.status = 'delivered') as cond_in_on,
(select count(*)
from customers c left join orders o 
on c.customer_id = o.customer_id
) as  count_star,
(
select count(o.order_id) from customers c left join orders o on c.customer_id=o.customer_id
) as count_orders

"""

out = q(sql)

assert list(out.columns) == ["cond_in_where", "cond_in_on", "count_star", "count_orders"]
assert len(out) == 1
assert out.round(2).values.tolist() == [[176, 190, 309, 300]]
print("OK")

OK


## Exercise 23: Mini Project — A Cohort Retention Table  *(mini project)*

Build the report that tells you whether customers come back.

Group customers by the **month of their first non-cancelled order** (their cohort), then count how many of that
cohort were active in each later period.

Columns:

- `cohort` — `'YYYY-MM'` of the first order
- `cohort_size` — distinct customers whose first order was that month
- `m1_3` — distinct customers from that cohort active 1 to 3 months later
- `m4_6` — active 4 to 6 months later
- `m7_plus` — active more than 6 months later
- `retained_pct` — `m1_3` as a percentage of `cohort_size`, rounded to 1

Restrict to cohorts from **2023**. Order by `cohort`.

Three steps, three CTEs: each customer's first order date; every order joined to that first date with the month
difference computed; then the counting. Use `COUNT(DISTINCT CASE WHEN ... THEN customer_id END)` so a customer
who ordered four times in a window is counted once.

In [ ]:
sql = """
-- Your SQL here
"""

out = q(sql)

assert list(out.columns) == ["cohort", "cohort_size", "m1_3", "m4_6", "m7_plus", "retained_pct"]
assert len(out) == 12
assert out.round(2).values.tolist() == [
    ["2023-01", 4, 1, 2, 4, 25],
    ["2023-02", 7, 3, 4, 7, 42.9],
    ["2023-03", 4, 2, 3, 4, 50],
    ["2023-04", 4, 1, 4, 4, 25],
    ["2023-05", 3, 0, 0, 2, 0],
    ["2023-06", 5, 2, 2, 5, 40],
    ["2023-07", 1, 0, 1, 1, 0],
    ["2023-08", 3, 0, 1, 2, 0],
    ["2023-09", 6, 4, 1, 4, 66.7],
    ["2023-10", 2, 1, 0, 2, 50],
    ["2023-11", 1, 0, 1, 1, 0],
    ["2023-12", 1, 0, 0, 0, 0],
]
print("OK")

## Exercise 24: Mini Project — Best Seller per Category per Month  *(mini project)*

The report a category manager wants: for each **category** and each **month of 2024**, the single best-selling
product by revenue, plus how much of that month's category revenue it accounted for.

Columns:

- `month`
- `category` — the category's `name`
- `product` — the winning product's `name`
- `revenue` — that product's revenue that month, rounded to 2
- `category_revenue` — the whole category's revenue that month, rounded to 2
- `share_pct` — the product's share of it, rounded to 1

Exclude cancelled orders. Order by `month`, then `category`. Limit to the first 20 rows.

This is the level in one query: a CTE at one row per product per category per month, a window function for the
category total on every row, `ROW_NUMBER` to pick the winner, and a filter outside the CTE that ranked it. Use
`ORDER BY revenue DESC, product` inside the `ROW_NUMBER` so ties resolve the same way every time.

In [60]:
sql = """
WITH product_month AS (
    SELECT
        strftime('%Y-%m', o.order_date) AS month,
        c.name AS category,
        p.name AS product,
        SUM(
            oi.quantity * oi.unit_price * (1 - oi.discount)
        ) AS revenue
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    JOIN products p
        ON oi.product_id = p.product_id
    JOIN categories c
        ON p.category_id = c.category_id
    WHERE o.status != 'cancelled'
      AND strftime('%Y', o.order_date) = '2024'
    GROUP BY
        month,
        c.name,
        p.name
),

ranked AS (
    SELECT
        month,
        category,
        product,
        revenue,

        SUM(revenue) OVER (
            PARTITION BY month, category
        ) AS category_revenue,

        ROW_NUMBER() OVER (
            PARTITION BY month, category
            ORDER BY revenue DESC, product
        ) AS rn

    FROM product_month
)

SELECT
    month,
    category,
    product,
    ROUND(revenue, 2) AS revenue,
    ROUND(category_revenue, 2) AS category_revenue,
    ROUND(100.0 * revenue / category_revenue, 1) AS share_pct
FROM ranked
WHERE rn = 1
ORDER BY month, category
LIMIT 20;
"""

out = q(sql)

assert list(out.columns) == ["month", "category", "product", "revenue", "category_revenue", "share_pct"]
assert len(out) == 20
assert out.round(2).values.tolist() == [
    ["2024-01", "Accessories", "Anchor 65W Charger", 6840, 14240, 48],
    ["2024-01", "Cameras", "Frame 50mm Lens", 14630, 14630, 100],
    ["2024-01", "Laptops", "Nimbus Air Laptop", 236000, 236000, 100],
    ["2024-01", "Monitors", "Clarity 27 QHD Monitor", 21500, 32120, 66.9],
    ["2024-01", "Phones", "Orbit Pro Phone", 48000, 48000, 100],
    ["2024-01", "Storage", "Vault 1TB SSD", 36480, 66340, 55],
    ["2024-01", "Wearables", "Tick Fitness Band", 2800, 2800, 100],
    ["2024-02", "Accessories", "Anchor 100W Charger", 5800, 11605, 50],
    ["2024-02", "Audio", "Rumble Bluetooth Speaker", 9000, 15800, 57],
    ["2024-02", "Cameras", "Frame Compact Camera", 30600, 30600, 100],
    ["2024-02", "Monitors", "Wide 34 Curved Monitor", 162400, 196600, 82.6],
    ["2024-02", "Phones", "Pixi Lite Phone", 12825, 12825, 100],
    ["2024-02", "Storage", "Vault 2TB SSD", 31920, 35820, 89.1],
    ["2024-02", "Wearables", "Tick Smartwatch", 9900, 12700, 78],
    ["2024-03", "Accessories", "Clack Mechanical Keyboard", 9800, 22515, 43.5],
    ["2024-03", "Audio", "Halo Studio Headset", 17010, 33510, 50.8],
    ["2024-03", "Cameras", "Frame Mirrorless Body", 163400, 178800, 91.4],
    ["2024-03", "Monitors", "Clarity 27 QHD Monitor", 107500, 218900, 49.1],
    ["2024-03", "Phones", "Orbit Pro Phone", 288000, 427050, 67.4],
    ["2024-03", "Storage", "Vault 2TB SSD", 10640, 14150, 75.2],
]
print("OK")

OK


## Self-Review Checklist

Check whether you can do each of these **without looking at the guide**.

- [ ] Join a table to itself and explain why the top of a hierarchy needs a `LEFT JOIN`.
- [ ] Say what `CROSS JOIN` is for, and name a report that needs it.
- [ ] Explain row fan-out to somebody, and fix a doubled `SUM` two different ways.
- [ ] State the grain of any query you have written — what one row means.
- [ ] Write an anti-join three ways and say which one is safe.
- [ ] Explain why `NOT IN` over a nullable subquery returns zero rows.
- [ ] Use a scalar subquery, a list subquery and a derived table, and say where each is allowed.
- [ ] Write a correlated subquery, and then replace it with a window function.
- [ ] Choose between `EXISTS` and a join for a given question.
- [ ] Use `UNION ALL`, `INTERSECT` and `EXCEPT`, and say where the `ORDER BY` goes.
- [ ] Write a three-step CTE chain and test each step on its own.
- [ ] Explain `OVER (PARTITION BY ...)` as `GROUP BY` that keeps its rows.
- [ ] Write the top-N-per-group shape from memory, including why the CTE is compulsory.
- [ ] Choose between `ROW_NUMBER`, `RANK` and `DENSE_RANK` for a given tie-handling requirement.
- [ ] Compute month-over-month change with `LAG`, and say why the first row is `NULL`.
- [ ] Write a running total and a 3-month moving average, and say what the default frame is.
- [ ] Explain why adding `ORDER BY` to a windowed `SUM` changes its meaning.
- [ ] Bucket a population with `NTILE` and say how ties are handled.
- [ ] Build a cross-tab with `SUM(CASE WHEN ...)` and say when `ELSE 0` is wrong.
- [ ] Compute whole months between two dates without a date type.
- [ ] Create a view and say what it does and does not store.
- [ ] Make a load idempotent with `ON CONFLICT ... DO UPDATE`.
- [ ] Say what a transaction guarantees and when you need one.
- [ ] Explain what moving a condition from `WHERE` to `ON` does to a `LEFT JOIN`.

## What Next

`sql-advanced` asks the question this level does not: why is the query slow? It covers query plans, indexes and
the rewrites that make a difference, recursive CTEs, transactions and isolation, schema design and star
schemas, and a set of classic interview problems built on everything here.

Before you move on, take the two mini projects and run them against a different date range. If they still make
sense and still add up, you have understood them rather than copied them.